# 실습 5: 고객 지원 Agent를 위한 GenAI Observability 자세히 살펴보기

## 개요

이 실습에서는 AgentCore Observability의 작동 방식과 AgentCore Runtime을 사용하지 않고 설정하는 방법을 알아봅니다.

## 추가할 기능

🧠 **AgentCore Observability 기능**:
- Amazon OpenTelemetry Python Instrumentation **설정**  
- Amazon CloudWatch GenAI Observability에서 Agent 추적 **시각화 및 분석**

## 튜토리얼 세부 정보

| 정보 | 세부 정보                                                          |
|-------------|------------------------------------------------------------------|
| **튜토리얼 유형** | 점진적 기능 향상                                          |
| **Agent 유형** | 단일 Agent                                                     |
| **Agentic 프레임워크** | Strands Agents                                                   |
| **LLM 모델** | Amazon Nova 2 Lite                                               |
| **튜토리얼 분야** | 고객 지원                                                 |
| **난이도** | 쉬움~보통                                                 |
| **사용 SDK** | Strands SDK, AgentCore Observability, CloudWatch, Bedrock, boto3 |

## 사전 요구 사항

- ✅ **먼저 실습 1을 완료해야 함** - 이 실습은 실습 1에서 만든 Agent를 기반으로 진행됩니다. 
- ✅ **Amazon CloudWatch에서 Transaction Search 활성화** - 처음 사용하는 경우 Bedrock AgentCore span과 추적을 보려면 CloudWatch Transaction Search를 활성화해야 합니다. 활성화 방법은 [문서](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html)를 참조하세요.

## 학습 목표

이 실습을 마치면 다음 작업을 수행할 수 있습니다.
- 공식 Amazon CloudWatch GenAI Observability Dashboard 사용


---

## 🚀 Agent에 Observability 추가하기


클라이언트 초기화

In [ ]:
import boto3
from botocore.exceptions import ClientError

session = boto3.Session()
region = session.region_name

logs_client = boto3.client("logs", region_name=region)
bedrock_client = boto3.client("bedrock", region_name=region)
sts_client = boto3.client("sts", region_name=region)

account_id = sts_client.get_caller_identity()["Account"]

`aws-opentelemetry-distro`가 설치되어 있는지 확인하세요.

In [ ]:
%pip install strands-agents boto3 aws-opentelemetry-distro -q

# 단계 2: Observability 환경 구성

Strands Agent의 Observability를 활성화하고 텔레메트리 데이터를 Amazon CloudWatch로 전송하려면 다음 환경 변수를 구성해야 합니다. 민감한 AWS 자격 증명을 코드와 분리하고 서로 다른 환경 간에 쉽게 전환할 수 있도록 `.env` 파일을 생성하여 설정을 안전하게 관리합니다.

필수 환경 변수:

| 변수 | 값 | 목적 |
|----------|-------|---------|
| `OTEL_PYTHON_DISTRO` | `aws_distro` | AWS Distro for OpenTelemetry(ADOT) 사용 |
| `OTEL_PYTHON_CONFIGURATOR` | `aws_configurator` | ADOT SDK용 AWS configurator 설정 |
| `OTEL_EXPORTER_OTLP_PROTOCOL` | `http/protobuf` | 내보내기 프로토콜 구성 |
| `OTEL_TRACES_EXPORTER` | `otlp` | 추적 exporter 구성 |
| `OTEL_EXPORTER_OTLP_LOGS_HEADERS` | `x-aws-log-group=<YOUR-LOG-GROUP>,x-aws-log-stream=<YOUR-LOG-STREAM>,x-aws-metric-namespace=<YOUR-NAMESPACE>` | CloudWatch 그룹으로 로그 전송 |
| `OTEL_RESOURCE_ATTRIBUTES` | `service.name=<YOUR-AGENT-NAME>` | Observability 데이터에서 Agent 식별 |
| `AGENT_OBSERVABILITY_ENABLED` | `true` | ADOT 파이프라인 활성화 |

또한 OpenTelemetry 계측 스크립트에서 사용할 수 있도록 `AWS_REGION`, `AWS_DEFAULT_REGION`, `AWS_ACCOUNT_ID` 환경 변수를 설정해야 합니다.

In [ ]:
log_group_name = "agents/customer-support-assistant-logs"  # 사용할 log group 이름
log_stream_name = "default"  # 사용할 log stream 이름

# 로그 그룹 생성
try:
    logs_client.create_log_group(logGroupName=log_group_name)
    print(f"✅ Log group '{log_group_name}' created successfully")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceAlreadyExistsException":
        print(f"ℹ️  Log group '{log_group_name}' already exists")
    else:
        print(f"❌ Error creating log group: {e}")

# 로그 스트림 생성
try:
    logs_client.create_log_stream(logGroupName=log_group_name, logStreamName=log_stream_name)
    print(f"✅ Log stream '{log_stream_name}' created successfully")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceAlreadyExistsException":
        print(f"ℹ️  Log stream '{log_stream_name}' already exists")
    else:
        print(f"❌ Error creating log stream: {e}")

In [ ]:
# .env 파일 생성
service_name = "customer-support-assistant-strands"

with open(".env", "w") as f:
    # AWS 구성
    f.write(f"AWS_REGION={region}\n")
    f.write(f"AWS_DEFAULT_REGION={region}\n")
    f.write(f"AWS_ACCOUNT_ID={account_id}\n")

    # AWS CloudWatch GenAI Observability용 OpenTelemetry 구성
    f.write("OTEL_PYTHON_DISTRO=aws_distro\n")
    f.write("OTEL_PYTHON_CONFIGURATOR=aws_configurator\n")
    f.write("OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf\n")
    f.write("OTEL_TRACES_EXPORTER=otlp\n")
    f.write(
        f"OTEL_EXPORTER_OTLP_LOGS_HEADERS=x-aws-log-group={log_group_name},x-aws-log-stream={log_stream_name},x-aws-metric-namespace=agents\n"
    )
    f.write(f"OTEL_RESOURCE_ATTRIBUTES=service.name={service_name}\n")
    f.write("AGENT_OBSERVABILITY_ENABLED=true\n")

In [ ]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv()

# OTEL 관련 환경 변수 표시
otel_vars = [
    "OTEL_PYTHON_DISTRO",
    "OTEL_PYTHON_CONFIGURATOR",
    "OTEL_EXPORTER_OTLP_PROTOCOL",
    "OTEL_EXPORTER_OTLP_LOGS_HEADERS",
    "OTEL_RESOURCE_ATTRIBUTES",
    "AGENT_OBSERVABILITY_ENABLED",
    "OTEL_TRACES_EXPORTER",
]

print("OpenTelemetry Configuration:\n")
for var in otel_vars:
    value = os.getenv(var)
    if value:
        print(f"{var}={value}")

# 단계 3: Strands Agent 정의

이제 이전과 동일한 Agent를 다시 정의합니다. 

추적이 생성되는 것을 보여 주기 위해 Agent에 간단한 인사 질의를 전달합니다.

또한 session ID가 등록되었는지 확인합니다.

In [ ]:
!cp lab_helpers/lab1_strands_agent.py customer_support_agent.py

In [ ]:
%%writefile -a customer_support_agent.py

import os
import argparse
from boto3.session import Session
from opentelemetry import baggage, context
from lab_helpers.utils import get_ssm_parameter

from strands import Agent
from strands.models import BedrockModel


def parse_arguments():
    parser = argparse.ArgumentParser(description="Customer Support Agent")
    parser.add_argument(
        "--session-id",
        type=str,
        required=True,
        help="Session ID to associate with this agent run",
    )
    return parser.parse_args()


def set_session_context(session_id):
    """추적 상관관계를 위해 OpenTelemetry baggage에 세션 ID를 설정합니다."""
    ctx = baggage.set_baggage("session.id", session_id)
    token = context.attach(ctx)
    print(f"Session ID '{session_id}' attached to telemetry context")
    return token


def main():
    # 명령줄 인수 구문 분석
    args = parse_arguments()

    # 텔레메트리용 세션 컨텍스트 설정
    context_token = set_session_context(args.session_id)

    # 리전 가져오기
    boto_session = Session()
    region = boto_session.region_name

    try:
        # 실습 1과 동일한 기본 Agent 생성
        MODEL = BedrockModel(
            model_id=MODEL_ID,
            temperature=0.3,
            region_name=region,
        )

        basic_agent = Agent(
            model=MODEL,
            tools=[
                get_product_info,
                get_return_policy,
            ],
            system_prompt=SYSTEM_PROMPT,
        )

        # 여행 조사 작업 실행
        query = """Greet the user and provide a financial advice."""

        result = basic_agent(query)
        print("Result:", result)

        print("✅ Agent executed successfully and trace was pushed to CloudWatch")
    finally:
        # 완료 후 컨텍스트 분리
        context.detach(context_token)


if __name__ == "__main__":
    main()

# 단계 4: AWS OpenTelemetry Python Distro

환경을 구성하고 Agent를 생성했으므로 Observability가 작동하는 방식을 알아보겠습니다. [AWS OpenTelemetry Python Distro](https://pypi.org/project/aws-opentelemetry-distro/)는 코드를 변경하지 않고도 텔레메트리 데이터를 캡처하도록 Strands Agent를 자동 계측합니다.

이 배포판은 다음 기능을 제공합니다.
- AgentCore Runtime 외부(예: EC2, Lambda 등)에서 호스팅되는 Strands Agent를 위한 **자동 계측**
- 원활한 CloudWatch 통합을 위한 **AWS 최적화 구성**  

### 계측된 Agent 실행

Strands Agent의 추적을 캡처하려면 Python을 직접 실행하는 대신 `opentelemetry-instrument` 명령을 사용합니다. 이 명령은 `.env` 파일의 환경 변수를 사용하여 계측을 자동으로 적용합니다.

```bash
opentelemetry-instrument python customer_support_assistant_agent.py
```

이 명령은 다음 작업을 수행합니다.

- .env 파일에서 OTEL 구성 로드
- Strands, Amazon Bedrock 호출, Agent 도구와 데이터베이스, Agent의 기타 요청 자동 계측
- CloudWatch로 추적 전송
- GenAI Observability 대시보드에서 Agent의 의사 결정 과정 시각화

In [ ]:
!opentelemetry-instrument python customer_support_agent.py --session-id "session-1234"

# 단계 5: GenAI Observability에서 확인 

Observability 구성을 마쳤으므로 AWS CloudWatch의 GenAI Observability 대시보드에서 추적을 확인합니다. CloudWatch - GenAI Observability - Bedrock AgentCore로 이동하세요.

#### Sessions 보기 페이지

![세션](images/sessions_lab5_observability.png)

#### Traces 보기 페이지
![추적](images/traces_lab5_observability.png)


## 축하합니다! 🎉

AgentCore Runtime 없이 **Strands Agent에 AgentCore Observability를 성공적으로 구현**했습니다!

### 완료한 작업

- ✅ **Observability**: Strands Agent가 텔레메트리 데이터를 Amazon CloudWatch로 전송하도록 구성
- ✅ **세션 관리**: 더 쉽게 디버깅할 수 있도록 세션별로 추적이 저장되는지 확인

## 다음 단계

AgentCore 기능을 더 추가할 준비가 되었나요? 다음 실습을 계속 진행하세요.

- **실습 6**: AgentCore Identity를 사용하여 외부 서비스에 안전하게 인증

## 리소스

- [AgentCore Observability 문서](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)
- [**공식 AgentCore Observability 샘플**](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/06-workshops/06-AgentCore-observability) ⭐

---

**훌륭합니다! 이제 프로덕션 환경에서 고객 지원 Agent의 성능을 추적, 디버깅, 모니터링할 수 있습니다! 🚀**
